# 4 · 3D no Blender (Cycles na GPU)

Roda um script de `Blender/scripts` (modelo procedural, cena, luz e câmera) no Blender 5.2.2 e renderiza com o
Cycles na GPU da VM (OptiX ou CUDA). Sai `<nome>.png` (render), `<nome>.glb` (para web e HyperFrames) e `<nome>.blend`, num .zip.

Pelo Painel (aba Blender → "renderizar no Colab") tudo é automático: o script sobe para a VM e o .zip volta para `Blender/saida`.
No navegador: envie o script e o `_estudio.py` para `/content` (ícone de pasta à esquerda), escreva o nome em SCRIPT e Executar tudo.

In [ ]:
#@title 1. Configuração
SCRIPT = "microfone.py"  #@param {type:"string"}
NOME = "microfone"  #@param {type:"string"}
PORCENTAGEM = 100  #@param {type:"integer"}
AMOSTRAS = 256  #@param {type:"integer"}

import os
SCRIPT = os.environ.get('SCRIPT', SCRIPT)
NOME = os.environ.get('NOME', NOME)
PORCENTAGEM = int(os.environ.get('PORCENTAGEM', PORCENTAGEM))
AMOSTRAS = int(os.environ.get('AMOSTRAS', AMOSTRAS))

In [ ]:
#@title 2. Drive e status para o Painel
from google.colab import drive
import json, os, time, threading, subprocess, collections
from datetime import datetime

RAIZ = '/content/drive/MyDrive/EditorVideos'
STATUS_DIR = f'{RAIZ}/_colab/status'

# A pasta EditorVideos vem do PC (Drive para computador). Se a montagem do Colab estiver velha ela
# não aparece; remonta e, se ainda faltar, para aqui em vez de criar uma "EditorVideos (1)" vazia.
# Pelo Colab CLI (Painel/colab-cli) o Drive já vem montado por `colab drivemount`: drive.mount() aqui
# ficaria esperando um navegador que não existe.
# Tarefas que não precisam do Drive (ex.: a voz pelo CLI) mandam EDITORVIDEOS_SEM_DRIVE=1: o status fica na VM
# e o Painel acompanha pelo log do CLI.
SEM_DRIVE = os.environ.get('EDITORVIDEOS_SEM_DRIVE') == '1'
if SEM_DRIVE:
    STATUS_DIR = '/content/_colab/status'
    print('Sem Drive nesta tarefa: o andamento vai pelo log do Colab CLI.')
elif os.environ.get('EDITORVIDEOS_CLI') and os.path.isdir('/content/drive/MyDrive'):
    print('Drive montado pelo Colab CLI.')
else:
    drive.mount('/content/drive', force_remount=True)
    if not os.path.isdir(f'{RAIZ}/HyperFrames'):
        time.sleep(10)
        drive.flush_and_unmount()
        drive.mount('/content/drive', force_remount=True)
if not SEM_DRIVE and not os.path.isdir(f'{RAIZ}/HyperFrames'):
    raise SystemExit('EditorVideos/HyperFrames não está visível neste Drive. Confira se o Colab está na mesma conta '
                     'do Drive do PC e se a sincronização terminou; depois Ambiente de execução → Desconectar e excluir.')

class Status:
    """Grava o andamento em _colab/status/<servico>.json. Batimento a cada 15 s enquanto roda."""
    def __init__(self, servico, notebook):
        os.makedirs(STATUS_DIR, exist_ok=True)
        self.arq = f'{STATUS_DIR}/{servico}.json'
        self.d = dict(servico=servico, notebook=notebook, estado='rodando', etapa='iniciando',
                      progresso=None, mensagem='', inicio=int(time.time() * 1000))
        self.log = collections.deque(maxlen=40)
        self._trava = threading.Lock()
        self._ultimo = 0
        self._vivo = True
        threading.Thread(target=self._batimento, daemon=True).start()
        self.gravar()

    def _gpu(self):
        try:
            out = subprocess.run(['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total',
                                  '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=5).stdout
            nome, uso, usada, total = [x.strip() for x in out.strip().splitlines()[0].split(',')]
            return dict(nome=nome, uso=int(uso), memoria_usada_mb=int(usada), memoria_total_mb=int(total))
        except Exception:
            return None

    def _ram(self):
        try:
            m = {}
            for linha in open('/proc/meminfo'):
                k, v = linha.split(':')
                m[k] = int(v.split()[0])
            return dict(usada_mb=(m['MemTotal'] - m['MemAvailable']) // 1024, total_mb=m['MemTotal'] // 1024)
        except Exception:
            return None

    def gravar(self):
        with self._trava:
            self.d.update(log=list(self.log), gpu=self._gpu(), ram=self._ram(), atualizado=int(time.time() * 1000))
            tmp = self.arq + '.tmp'
            with open(tmp, 'w') as f:
                json.dump(self.d, f, ensure_ascii=False)
            os.replace(tmp, self.arq)
            self._ultimo = time.time()

    def _batimento(self):
        while self._vivo:
            time.sleep(15)
            if self._vivo:
                self.gravar()

    def __call__(self, etapa=None, progresso=None, mensagem=None, linha=None):
        mudou = etapa is not None and etapa != self.d['etapa']
        if etapa is not None: self.d['etapa'] = etapa
        if progresso is not None: self.d['progresso'] = round(progresso, 1)
        if mensagem is not None: self.d['mensagem'] = mensagem
        if linha:
            self.log.append(f'{datetime.now():%H:%M:%S} {linha}')
            print(linha)
        # Não martela o Drive: no máximo uma gravação a cada 3 s, salvo mudança de etapa.
        if mudou or linha or time.time() - self._ultimo > 3:
            self.gravar()

    def fim(self, estado='concluido', mensagem=''):
        self._vivo = False
        self.d.update(estado=estado, mensagem=mensagem, etapa='fim' if estado == 'concluido' else self.d['etapa'])
        if estado == 'concluido': self.d['progresso'] = 100
        self.gravar()

print('Drive montado. Status em', STATUS_DIR)

In [ ]:
#@title 3. Instalar o Blender (oficial, conferido)
st = Status('blender', '4_blender')

import hashlib, subprocess, urllib.request
PASTA = '/content/blender-5.2.2-linux-x64'
BLENDER = f'{PASTA}/blender'
if not os.path.exists(BLENDER):
    st('baixando', 2, 'Blender 5.2.2', 'baixando o Blender (uns 380 MB)')
    pacote = '/content/blender.tar.xz'
    urllib.request.urlretrieve('https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz', pacote)
    h = hashlib.sha256()
    with open(pacote, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 20), b''):
            h.update(bloco)
    if h.hexdigest() != '84098912789dc450e95697c4184fb8a90acbe5111c2ba4aede3fecb57806a168':
        raise SystemExit(f'SHA-256 do Blender não confere ({h.hexdigest()}); download interrompido ou adulterado.')
    st('extraindo', 8, linha='SHA-256 conferido')
    subprocess.run(['tar', '-xf', pacote, '-C', '/content'], check=True)
    os.remove(pacote)
    # Bibliotecas de sistema que o Blender pede mesmo sem interface.
    subprocess.run('apt-get -qq install -y libxkbcommon0 libsm6 libxi6 libxxf86vm1 libgl1 > /dev/null', shell=True)
st('instalado', 12, linha=subprocess.run([BLENDER, '--version'], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
#@title 4. Renderizar
import re, shutil, time
SAIDA = f'/content/blender/saida/{NOME}'
os.makedirs(SAIDA, exist_ok=True)
caminho = SCRIPT if os.path.isabs(SCRIPT) else f'/content/{SCRIPT}'
try:
    if not os.path.exists(caminho):
        raise FileNotFoundError(f'Script não encontrado: {caminho}')
    st('renderizando', 15, SCRIPT, f'{PORCENTAGEM}% · {AMOSTRAS} amostras na GPU')
    t0 = time.time()
    proc = subprocess.Popen([BLENDER, '-b', '--factory-startup', '--offline-mode', '-P', caminho, '--', SAIDA, str(PORCENTAGEM), str(AMOSTRAS)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, 'EDITORVIDEOS_GPU': '1'})
    ultimo = 0
    for linha in proc.stdout:
        linha = linha.rstrip()
        amostra = re.search(r'Sample (\d+)/(\d+)', linha)
        if amostra and time.time() - ultimo > 5:
            ultimo = time.time()
            a, b = int(amostra.group(1)), int(amostra.group(2))
            st(f'amostra {a}/{b}', 15 + a / b * 80)
            print(f'[3d] amostra {a}/{b} ({a / b * 100:.0f}%)', flush=True)
        elif linha.startswith('[3d]') or 'Error' in linha or 'Traceback' in linha:
            st(linha=linha)
    if proc.wait() != 0:
        raise RuntimeError(f'O Blender saiu com código {proc.returncode} (veja o log acima).')
    shutil.make_archive(f'/content/blender/{NOME}', 'zip', SAIDA)
    st(linha=f'ok: {NOME}.zip em {time.time() - t0:.0f} s ({", ".join(sorted(os.listdir(SAIDA)))})')
    st.fim('concluido', f'{NOME}: render em {time.time() - t0:.0f} s')
except Exception as e:
    st(linha=f'ERRO: {e}')
    st.fim('erro', str(e))
    raise